In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from simulations.benchmarks.hf import hf
from simulations.molecules import MoleculeSimulator
from solvers.evc_solver import EVCSolver

In [4]:
from src.utils.molecule_utils import compute_cc
from src.utils.procrustes_utils import (
    compute_procrustes_matrices,
    compute_cc_with_procrustes
)

## Simulator

In [5]:
hf_simulator = MoleculeSimulator(
    molecule_fun=hf,
    basis="cc-pVTZ",
    coord_scale=0.1,
    verbose=0,
)

### Reference molecule

In [6]:
include_kwargs = {
    "include_integrals": True,
    "include_hartree_fock": True,
    "include_cc": False,
    "include_coordinates": False,
    "include_all": False
}

hf_reference = hf_simulator.simulate(molecule_kwargs={"bond_distance": 1.75, "perturb": False}, **include_kwargs)

In [7]:
hf_reference["positions"]

array([[0.  , 0.  , 0.  ],
       [1.75, 0.  , 0.  ]], dtype=float32)

In [8]:
reference_determinant = hf_reference["determinant"]
reference_determinant.shape

(44, 44)

In [9]:
reference_overlap = hf_reference["overlaps"]
reference_overlap.shape

(44, 44)

### Sample and target molecules

In [10]:
sample_molecules = hf_simulator.sample(10, include_kwargs={"include_all": False})

Generating samples: 100%|██████████| 10/10 [00:00<00:00, 815.00it/s]


In [11]:
sample_procrustes_matrices = compute_procrustes_matrices(
    batched_atoms=sample_molecules["atoms"],
    batched_positions=sample_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing procrustes: 100%|██████████| 10/10 [00:07<00:00,  1.42it/s]


In [12]:
target_molecules = hf_simulator.sample(81, include_kwargs={"include_all": False})

Generating samples: 100%|██████████| 81/81 [00:00<00:00, 905.74it/s]


In [13]:
target_procrustes_matrices = compute_procrustes_matrices(
    batched_atoms=target_molecules["atoms"],
    batched_positions=target_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing procrustes: 100%|██████████| 81/81 [00:56<00:00,  1.43it/s]


In [14]:
# This replaces setup_sample().
procrustes_cc = compute_cc_with_procrustes(
    batched_atoms=sample_molecules["atoms"],
    batched_positions=sample_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing CCSD: 100%|██████████| 10/10 [00:23<00:00,  2.33s/it]


In [35]:
procrustes_cc.keys()

dict_keys(['t1', 't2', 'energies'])

In [36]:
for k, v in procrustes_cc.items():
    print(f"{k}: {v.shape}")

t1: (10, 5, 39)
t2: (10, 5, 5, 39, 39)
energies: (10,)


In [38]:
evc_solver = EVCSolver(
    t1s=procrustes_cc["t1"],
    t2s=procrustes_cc["t2"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap
)

TypeError: EVCSolver.__init__() missing 3 required positional arguments: 'all_x', 'molecule_func', and 'basis'